# Weather Lakehouse — Bronze to Silver Incremental Processing

This notebook discovers raw weather JSON files in ADLS Gen2 Bronze, identifies new or modified files, transforms hourly observations with PySpark, removes duplicate business keys, and incrementally upserts records into the Silver Delta table.

**Processing flow**

1. Configure Azure Storage and Delta table settings.
2. Create or verify the file-processing control table.
3. Discover Bronze JSON files and select the weather datasets.
4. Identify files that are new, modified, or previously failed.
5. Parse JSON files and build the hourly source DataFrame.
6. Validate and deduplicate records.
7. MERGE valid records into the Silver Delta table.
8. Update the separate control table with per-file SUCCESS or FAILED status.
9. Validate the results and monitor processing.

**Important:** Run the operational sections from top to bottom. The diagnostic, failure-simulation, and control-table reset sections at the end are optional and should not be run during normal processing.

## 1. Imports and Configuration

Set the storage account, Bronze container, folder prefix, SAS token, Silver target table, and shared column definitions.

The SAS token is entered securely and should never be printed or committed to source control.


In [0]:
import requests
import xml.etree.ElementTree as ET
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# Databricks Job parameters
dbutils.widgets.text("storage_account", "")
dbutils.widgets.text("bronze_container", "")
dbutils.widgets.text("blob_prefix", "")

STORAGE_ACCOUNT = dbutils.widgets.get("storage_account").strip()
CONTAINER = dbutils.widgets.get("bronze_container").strip()
PREFIX = dbutils.widgets.get("blob_prefix").strip().strip("/")

# Retrieve SAS token from Databricks Secret Scope
SAS_TOKEN = dbutils.secrets.get(
    scope="weather_lakehouse",
    key="bronze_sas_token"
).strip().lstrip("?")

# Silver target
TARGET_TABLE = "workspace.default.weather_hourly_silver"

# Storage endpoint
BASE_CONTAINER_URL = (
    f"https://{STORAGE_ACCOUNT}.blob.core.windows.net/{CONTAINER}"
)

# Weather columns
WEATHER_COLUMNS = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "wind_speed_10m"
]

# Silver merge key
MERGE_KEYS = ["location", "weather_timestamp"]

print("Configuration initialized.")
print(f"Storage account: {STORAGE_ACCOUNT}")
print(f"Container: {CONTAINER}")
print(f"Bronze prefix: {PREFIX}")
print(f"Silver target: {TARGET_TABLE}")

## 2. Create the Bronze File-Processing Control Table

The control table stores file-level processing metadata independently of the Silver weather records.

- `SUCCESS`: the file was parsed and its records were successfully merged into Silver.
- `FAILED`: parsing or the Silver MERGE failed; the file remains eligible for retry.


In [0]:
# CONTROL_TABLE = "workspace.default.bronze_file_processing_log"

# spark.sql(f"""
# CREATE TABLE IF NOT EXISTS {CONTROL_TABLE} (
#     blob_path STRING,
#     last_modified TIMESTAMP,
#     etag STRING,
#     file_size BIGINT,
#     processing_status STRING,
#     processed_at TIMESTAMP,
#     error_message STRING
# )
# USING DELTA
# """)

# print(f"Control table ready: {CONTROL_TABLE}")

## 3. Discover JSON Files in Bronze

List blobs under the configured Bronze prefix using the Azure Blob Storage REST API. The listing captures each file's path, last-modified timestamp, ETag, and size.


In [0]:
from urllib.parse import urlencode

def list_bronze_files():
    files = []
    marker = None

    while True:
        params = {
            "restype": "container",
            "comp": "list",
            "prefix": PREFIX
        }

        if marker:
            params["marker"] = marker

        # Include SAS authentication on every request
        url = f"{BASE_CONTAINER_URL}?{SAS_TOKEN}&{urlencode(params)}"

        response = requests.get(url, timeout=60)
        response.raise_for_status()

        root = ET.fromstring(response.content)

        for blob in root.findall(".//{*}Blob"):
            name = blob.findtext("{*}Name")
            properties = blob.find("{*}Properties")

            if not name or properties is None:
                continue

            if not name.lower().endswith(".json"):
                continue

            files.append({
                "blob_path": name,
                "last_modified": properties.findtext("{*}Last-Modified"),
                "etag": properties.findtext("{*}Etag"),
                "file_size": properties.findtext("{*}Content-Length")
            })

        marker = root.findtext("{*}NextMarker")

        if not marker:
            break

    return files


bronze_files = list_bronze_files()

print(f"Total JSON files found: {len(bronze_files)}")

for file in bronze_files[:10]:
    print(file)

## 4. Select Weather Datasets

Keep only the intended archive and forecast JSON paths. Exclude unrelated JSON files or metadata folders.


In [0]:
# Keep only the two intended ingestion datasets
weather_files = [
    file for file in bronze_files
    if file["blob_path"].startswith((
        "weather/archive/",
        "weather/forecast/"
    ))
]

archive_files = [
    file for file in weather_files
    if file["blob_path"].startswith("weather/archive/")
]

forecast_files = [
    file for file in weather_files
    if file["blob_path"].startswith("weather/forecast/")
]

print("Archive files:", len(archive_files))
print("Forecast files:", len(forecast_files))
print("Total weather files:", len(weather_files))

## 5. Normalize Bronze File Metadata

Convert Azure `Last-Modified` values to UTC-naive timestamps for Spark and cast file sizes to integers. Build the metadata DataFrame used by incremental file detection.


In [0]:
from datetime import datetime, timezone

# Convert Last-Modified values safely
for file in weather_files:
    value = file["last_modified"]

    if isinstance(value, str):
        value = datetime.strptime(
            value,
            "%a, %d %b %Y %H:%M:%S GMT"
        ).replace(tzinfo=timezone.utc)

    if value.tzinfo is not None:
        value = value.astimezone(timezone.utc).replace(tzinfo=None)

    file["last_modified"] = value
    file["file_size"] = int(file["file_size"] or 0)

# Create metadata DataFrame
weather_files_df = spark.createDataFrame(weather_files)

print("Weather metadata prepared:", weather_files_df.count())

display(weather_files_df)

In [0]:
from datetime import datetime, timezone

# Convert Last-Modified values safely
for file in weather_files:
    value = file["last_modified"]

    if isinstance(value, str):
        value = datetime.strptime(
            value,
            "%a, %d %b %Y %H:%M:%S GMT"
        ).replace(tzinfo=timezone.utc)

    if value.tzinfo is not None:
        value = value.astimezone(timezone.utc).replace(tzinfo=None)

    file["last_modified"] = value
    file["file_size"] = int(file["file_size"] or 0)

# Create metadata DataFrame
weather_files_df = spark.createDataFrame(weather_files)

print("Weather metadata prepared:", weather_files_df.count())

display(weather_files_df)

## 6. Detect New, Modified, and Retryable Files

Compare current Bronze metadata with files marked `SUCCESS` in the control table.

A file is selected when:
- no SUCCESS record exists for its path;
- its ETag has changed;
- its last-modified timestamp has changed; or
- its file size has changed.

Files marked `FAILED` are intentionally not excluded, so they are selected again for retry.


In [0]:
processed_files_df = (
    spark.table(CONTROL_TABLE)
    .filter(F.col("processing_status") == "SUCCESS")
    .select(
        "blob_path",
        "last_modified",
        "etag",
        "file_size"
    )
)

new_or_modified_files_df = (
    weather_files_df.alias("b")
    .join(
        processed_files_df.alias("p"),
        on="blob_path",
        how="left"
    )
    .filter(
        F.col("p.blob_path").isNull()
        | (F.col("b.etag") != F.col("p.etag"))
        | (F.col("b.last_modified") != F.col("p.last_modified"))
        | (F.col("b.file_size") != F.col("p.file_size"))
    )
    .select("b.*")
)

print("Files requiring processing:", new_or_modified_files_df.count())

display(new_or_modified_files_df)

## 7. Optional Diagnostic — Inspect a Sample Bronze JSON File

This section is for understanding the JSON structure. It prefers a pending file, but falls back to any discovered weather file when there are no pending files. It does not modify Bronze, Silver, or the control table.


In [0]:
from urllib.parse import parse_qsl

# Prefer a file pending processing; otherwise inspect any discovered weather file.
sample_row = new_or_modified_files_df.select("blob_path").first()

if sample_row is None:
    print("No pending files. Selecting a sample from all discovered Bronze files.")
    sample_row = weather_files_df.select("blob_path").first()

if sample_row is None:
    raise ValueError("No Bronze JSON files found.")

sample_file = sample_row["blob_path"]

# Download the file using the existing SAS token.
sas_params = dict(parse_qsl(SAS_TOKEN))
file_url = f"{BASE_CONTAINER_URL}/{sample_file}"

response = requests.get(file_url, params=sas_params, timeout=60)
response.raise_for_status()

sample_json = response.json()

print("Sample file:", sample_file)
print("Root keys:", list(sample_json.keys()))

if "hourly" in sample_json:
    print("Hourly keys:", list(sample_json["hourly"].keys()))
    for key, values in sample_json["hourly"].items():
        print(f"{key}: {len(values) if isinstance(values, list) else 'not a list'}")


In [0]:
import re

sample_path = sample_file

location_match = re.search(
    r"location=([^/]+)",
    sample_path
)

location = location_match.group(1) if location_match else None

print("Location:", location)
print("First timestamp:", sample_json["hourly"]["time"][0])
print("Last timestamp:", sample_json["hourly"]["time"][-1])
print("Timezone:", sample_json["timezone"])
print("UTC offset seconds:", sample_json["utc_offset_seconds"])

## 8. Parse Selected Bronze Files into Hourly Records

Download each selected JSON file, extract the location from its path, validate the hourly arrays, and convert each hourly observation into a record.

Parsing errors are captured per file in `file_errors`. A failed file does not prevent the other files from being parsed.


In [0]:
from datetime import datetime, timezone
import re

records = []
file_errors = []

for file in new_or_modified_files_df.collect():
    blob_path = file["blob_path"]

    try:
        location_match = re.search(r"location=([^/]+)", blob_path)
        if not location_match:
            raise ValueError("Location not found in file path")

        location = location_match.group(1)

        response = requests.get(
            f"{BASE_CONTAINER_URL}/{blob_path}",
            params=sas_params,
            timeout=60
        )
        response.raise_for_status()

        data = response.json()
        hourly = data["hourly"]

        required_keys = [
            "time",
            "temperature_2m",
            "relative_humidity_2m",
            "precipitation",
            "wind_speed_10m"
        ]

        lengths = [len(hourly[key]) for key in required_keys]

        if len(set(lengths)) != 1:
            raise ValueError(f"Hourly array length mismatch: {lengths}")

        for i in range(lengths[0]):
            records.append({
                "location": location,
                "weather_timestamp": datetime.strptime(
                    hourly["time"][i], "%Y-%m-%dT%H:%M"
                ),
                "temperature_2m": hourly["temperature_2m"][i],
                "relative_humidity_2m": hourly["relative_humidity_2m"][i],
                "precipitation": hourly["precipitation"][i],
                "wind_speed_10m": hourly["wind_speed_10m"][i],
                "ingested_at_utc": datetime.now(timezone.utc).replace(tzinfo=None),
                "ingestion_date": datetime.now(timezone.utc).date(),
                "source_file": blob_path,
                "source_last_modified": file["last_modified"]
            })

    except Exception as e:
        file_errors.append({
            "blob_path": blob_path,
            "error": str(e)
        })

print("Files selected:", new_or_modified_files_df.count())
print("Files parsed successfully:", new_or_modified_files_df.count() - len(file_errors))
print("Hourly records created:", len(records))
print("Files with errors:", len(file_errors))

if file_errors:
    display(spark.createDataFrame(file_errors))

## 9. Create the Raw PySpark Source DataFrame

Apply an explicit schema to the parsed records. The source includes the weather measures, ingestion metadata, and source lineage columns.


In [0]:
from pyspark.sql.types import (
    StructType, StructField, StringType,
    TimestampType, DoubleType, DateType
)

source_schema = StructType([
    StructField("location", StringType(), False),
    StructField("weather_timestamp", TimestampType(), False),
    StructField("temperature_2m", DoubleType(), True),
    StructField("relative_humidity_2m", DoubleType(), True),
    StructField("precipitation", DoubleType(), True),
    StructField("wind_speed_10m", DoubleType(), True),
    StructField("ingested_at_utc", TimestampType(), False),
    StructField("ingestion_date", DateType(), False),
    StructField("source_file", StringType(), False),
    StructField("source_last_modified", TimestampType(), False)
])

source_df = spark.createDataFrame(records, schema=source_schema)

print("Total records:", source_df.count())
source_df.printSchema()

display(source_df.limit(10))

## 10. Data Quality — Identify Duplicate Business Keys

Check for duplicate `(location, weather_timestamp)` keys and quantify the extra records before deduplication.


In [0]:
# Find duplicate business keys
duplicate_keys_df = (
    source_df
    .groupBy("location", "weather_timestamp")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_key_count = duplicate_keys_df.count()

print("Duplicate business keys:", duplicate_key_count)

# Calculate how many extra records duplicates represent
duplicate_extra_records = (
    duplicate_keys_df
    .select(F.sum(F.col("count") - 1))
    .first()[0]
)

print("Extra duplicate records:", duplicate_extra_records or 0)

display(duplicate_keys_df.orderBy("location", "weather_timestamp"))

## 11. Deduplicate Hourly Weather Records

Keep one record per `(location, weather_timestamp)` key. When duplicates exist, retain the record from the most recently modified source file, using the source path as a deterministic tie-breaker.


In [0]:
dedup_window = (
    Window
    .partitionBy("location", "weather_timestamp")
    .orderBy(
        F.col("source_last_modified").desc(),
        F.col("source_file").desc()
    )
)

deduplicated_df = (
    source_df
    .withColumn("row_num", F.row_number().over(dedup_window))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

print("Records before deduplication:", source_df.count())
print("Records after deduplication:", deduplicated_df.count())
print(
    "Records removed:",
    source_df.count() - deduplicated_df.count()
)

# Verify no duplicate business keys remain
remaining_duplicates = (
    deduplicated_df
    .groupBy("location", "weather_timestamp")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Remaining duplicate keys:", remaining_duplicates)

## 12. Prepare and Inspect the Silver MERGE Source

Review the incoming deduplicated records and compare their schema and count with the existing Silver table.

The actual MERGE uses `deduplicated_df`, including both source lineage columns.


In [0]:
silver_source_df = deduplicated_df.select(
    "location",
    "weather_timestamp",
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "wind_speed_10m",
    "ingested_at_utc",
    "ingestion_date"
)

print("Silver source records:", silver_source_df.count())
silver_source_df.printSchema()

In [0]:
print("Silver table schema:")
spark.table(TARGET_TABLE).printSchema()

print("Current Silver record count:", spark.table(TARGET_TABLE).count())

print("Incoming source record count:", silver_source_df.count())

## 13. Incremental Delta MERGE into Silver

Upsert records using `(location, weather_timestamp)` as the business key.

- Insert new keys.
- Update existing keys only when one or more weather measures have changed.
- Refresh ingestion and source-lineage metadata when an update is required.
- Retain the MERGE outcome so the control table can record the correct file statuses.


In [0]:
silver_merge_succeeded = False
silver_merge_error = None

try:
    update_condition = " OR ".join(
        [f"NOT (t.{c} <=> s.{c})" for c in WEATHER_COLUMNS]
    )

    update_set = {
        c: f"s.{c}"
        for c in WEATHER_COLUMNS + [
            "ingested_at_utc",
            "ingestion_date",
            "source_file",
            "source_last_modified"
        ]
    }

    (
        DeltaTable.forName(spark, TARGET_TABLE)
        .alias("t")
        .merge(
            deduplicated_df.alias("s"),
            """
            t.location = s.location
            AND t.weather_timestamp = s.weather_timestamp
            """
        )
        .whenMatchedUpdate(
            condition=update_condition,
            set=update_set
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    silver_merge_succeeded = True
    print("Silver MERGE completed successfully.")

except Exception as e:
    silver_merge_error = str(e)
    print("Silver MERGE failed:", silver_merge_error)
    # Do not stop the notebook here. Cell 25 records the affected files as FAILED.


## 14. Validate the Silver Delta Table

Check total records, distribution by location, duplicate business keys, and sample rows after the MERGE.


In [0]:
silver_df = spark.table(TARGET_TABLE)

print("Total Silver records:", silver_df.count())

print("Records by location:")
silver_df.groupBy("location").count().orderBy("location").show()

print("Duplicate business keys:")
silver_df.groupBy("location", "weather_timestamp") \
    .count() \
    .filter("count > 1") \
    .show()

print("Sample Silver records:")
silver_df.orderBy(
    "location", "weather_timestamp"
).show(10, truncate=False)

## 15. Update the Separate Bronze Processing Control Table

Snapshot the selected file metadata before updating the control table. Then assign each file its individual status:

- Parsing error → `FAILED` with the parsing error.
- Parsing succeeded and Silver MERGE succeeded → `SUCCESS`.
- Parsing succeeded but Silver MERGE failed → `FAILED` with the MERGE error.

This cell runs after the Silver MERGE and records the outcome for each selected file.


In [0]:
selected_file_rows = new_or_modified_files_df.collect()

error_by_file = {
    item["blob_path"]: item["error"]
    for item in file_errors
}

control_rows = []
for file in selected_file_rows:
    blob_path = file["blob_path"]

    if blob_path in error_by_file:
        status = "FAILED"
        error_message = error_by_file[blob_path]
    elif silver_merge_succeeded:
        status = "SUCCESS"
        error_message = None
    else:
        status = "FAILED"
        error_message = (
            silver_merge_error
            or "Silver MERGE did not complete successfully"
        )

    control_rows.append((
        blob_path,
        file["last_modified"],
        file["etag"],
        int(file["file_size"] or 0),
        status,
        None,
        error_message
    ))

control_schema = """
    blob_path STRING,
    last_modified TIMESTAMP,
    etag STRING,
    file_size BIGINT,
    processing_status STRING,
    processed_at TIMESTAMP,
    error_message STRING
"""

if control_rows:
    control_source_df = (
        spark.createDataFrame(control_rows, schema=control_schema)
        .withColumn("processed_at", F.current_timestamp())
    )

    (
        DeltaTable.forName(spark, CONTROL_TABLE)
        .alias("t")
        .merge(
            control_source_df.alias("s"),
            "t.blob_path = s.blob_path"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print("Bronze processing log updated.")
    print("Files recorded:", len(control_rows))
    print("SUCCESS files:", sum(1 for row in control_rows if row[4] == "SUCCESS"))
    print("FAILED files:", sum(1 for row in control_rows if row[4] == "FAILED"))
else:
    print("No new or modified Bronze files to record.")

# Display the exact statuses prepared for this processing batch.
if control_rows:
    display(control_source_df.orderBy("processing_status", "blob_path"))


## 16. Operational Monitoring

Review the query plan and the current Bronze/control-table counts. These cells are read-only diagnostics.


In [0]:
print("NEW_OR_MODIFIED_FILES_DF PLAN")
new_or_modified_files_df.explain(mode="extended")

In [0]:
print("Bronze files:", weather_files_df.count())

print("Control table records:",
      spark.table(CONTROL_TABLE).count())

print("SUCCESS records:",
      spark.table(CONTROL_TABLE)
      .filter(F.col("processing_status") == "SUCCESS")
      .count())

print("Currently detected files:",
      new_or_modified_files_df.count())